<a href="https://colab.research.google.com/github/ms24191-PN/IDC-AQIupgrade/blob/main/Complete_data_visualization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Loading the data

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("rohanrao/air-quality-data-in-india")

print("Path to dataset files:", path)

import os
print(os.listdir(path))

In [ ]:
import pandas as pd

file_path = os.path.join(path, 'city_day.csv')
data = pd.read_csv(file_path)
data = data.dropna(subset=['AQI']) #Deleting the null values
#print (data_final.head())

print('DATA.SHAPE : ',"\n", data.shape, "\n", 'DATA.COLUMNS : ', "\n", data.columns, "\n",'DATA.HEAD : ', "\n",  data.head, "\n",'DATA.INFO', "\n",  data.info, "\n",'DATA.DESCRIBE',"\n", data.describe)

# Filling NaN values of Pollutant

In [ ]:
pollutants = [
    'PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3',
    'CO', 'SO2', 'O3', 'Benzene', 'Toluene', 'Xylene'
]

# Replace missing values with the median of each pollutant
data[pollutants] = data[pollutants].fillna(data[pollutants].median())

# Check missing values after imputation
print(data[pollutants].isnull().sum())


# Data Preprocessing

In [ ]:
print('NULL VALUES FOR RESPECTIVE COLUMNS :','\n', data.isnull().sum(),'\n', data.duplicated().sum())
print('DUPLICATED VALUES IN THE DATA :','\n',data.duplicated().sum()) #Checking the duplicate values
print('NO OF UNIQUE CITIES : ','\n',data['City'].unique())  #Checking the inconsistent city/station names.
print('NO VALUES PER CITIES : ', '\n', data['City'].value_counts())
print('DTYPE OD DATE : ', '\n',data['Date'].dtype)

## Data feature addition

### Defining the Season function

In [ ]:
def get_season(month):
  #Standard seson month in NH are used here
  if month in [12,1,2]:
    return "Winter"
  elif month in [3,4,5]:
    return "Spring"
  elif month in [6,7,8]:
    return "Summer"
  else:
    return "Monsoon"

In [ ]:
data['Date'] = pd.to_datetime(data['Date'])   #COnverting the dtype from object to datetime
data['Year'] = data['Date'].dt.year
data['Month']= data['Date'].dt.month     #Creating month data feature
data['Weekday'] = data['Date'].dt.day_name()   #Creating weekday data feature
data['Day'] = data['Date'].dt.day
data['Season'] = data['Month'].apply(get_season)     #Creating season  data feature
data.head()
print(data.groupby('Season')['AQI'].mean())
data['Winter'] = data['Month'].isin([12,1,2]).astype(int) #Here 1 -> Winter month and 0 -> other month
print(data.groupby('Winter')['AQI'].mean())      #Creating Winter indicator data feature

AQI_data = data.copy()

Downloading the Feature and Target data as csv

In [ ]:
#from google.colab import files

#data.to_csv('AQI_data.csv', index=False)
#files.download('AQI_data.csv')

# Visualization

## City-wise AQI

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

data = AQI_data.copy()
#let calculate the average of the year
avg_year_aqi = (data.groupby(['City','Year'])['AQI'].mean().reset_index())

cities = sorted(avg_year_aqi['City'].unique())
colors = [
    "#E6194B", "#3CB44B", "#4363D8", "#F58231",
    "#911EB4", "#42D4F4", "#F032E6", "#BFEF45",
    "#FABED4", "#469990", "#DCBEFF", "#9A6324",
    "#800000", "#AAFFC3", "#000075", "#A9A9A9",
    "#FF0000", "#00FF00", "#0000FF", "#FFD700",
    "#FF1493", "#00CED1", "#FF8C00", "#4B0082",
    "#8B4513", "#2F4F4F"
]
plt.figure(figsize = (16,8))

for city , color in zip(cities, colors):
  city_data = avg_year_aqi[avg_year_aqi['City'] == city].sort_values('Year')
  plt.plot(city_data['Year'], city_data['AQI'], marker = 'o', linewidth =2, markersize = 5, label= city, color = color)

plt.title('Annual aqi of the city-wise', fontsize = 16)
plt.xlabel('Year')
plt.ylabel('Average AQI')

plt.xticks(sorted(avg_year_aqi['Year'].unique()), rotation = 45)
plt.legend(title = 'City', bbox_to_anchor = (1.02,1), loc ='upper left')
plt.grid(True, alpha = 0.3)
plt.tight_layout()
plt.show()

For top 10 Cities


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Counting the available AQI data points for each city
city_counts = (data.dropna(subset=['AQI']).groupby('City')['AQI'].count().sort_values(ascending=False))

#Selecting top 10 cities with most AQI data points
top_10_cities = city_counts.head(10).index.tolist()

#Filtering data for selected cities
top_data = data[data['City'].isin(top_10_cities)].copy()

#Calculating the annual average AQI for each city
yearly_avg_aqi = (top_data.groupby(['City', 'Year'])['AQI'].mean().reset_index())

#Plotting all 10 cities data in a graph
colors = sns.color_palette('husl', n_colors=len(top_10_cities))

plt.figure(figsize=(16, 8))

for city, color in zip(top_10_cities, colors):

    city_data = yearly_avg_aqi[yearly_avg_aqi['City'] == city].sort_values('Year')

    plt.plot(
        city_data['Year'],
        city_data['AQI'],
        marker='o',
        linewidth=2,
        markersize=5,
        label=city,
        color=color
    )

plt.title('Annual Average AQI: Top 10 Cities',fontsize=16)
plt.xlabel('Year', fontsize=12)
plt.ylabel('Annual Average AQI', fontsize=12)

plt.xticks(sorted(yearly_avg_aqi['Year'].unique()),rotation=45)
plt.legend(title='City',bbox_to_anchor=(1.02, 1),loc='upper left')

plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


#Calculate overall average AQI for each selected city
city_avg_aqi = (top_data.groupby('City').agg(Data_Points=('AQI', 'count'),Average_AQI=('AQI', 'mean')).reset_index())

# Rank cities by average AQI
city_avg_aqi = city_avg_aqi.sort_values('Average_AQI',ascending=False).reset_index(drop=True)

city_avg_aqi['Average_AQI'] = city_avg_aqi['Average_AQI'].round(2)

# Display the complete ranking table
print("AQI Ranking of the Top 10 Cities")
display(city_avg_aqi)

# 5. Identify the highest and lowest average AQI cities
highest_aqi_city = city_avg_aqi.loc[city_avg_aqi['Average_AQI'].idxmax()]

lowest_aqi_city = city_avg_aqi.loc[city_avg_aqi['Average_AQI'].idxmin()]

# Create summary table
summary_table = pd.DataFrame([
    {
        'Category': 'Highest Average AQI',
        'City': highest_aqi_city['City'],
        'Average AQI': highest_aqi_city['Average_AQI'],
        'Data Points': highest_aqi_city['Data_Points']
    },
    {
        'Category': 'Lowest Average AQI',
        'City': lowest_aqi_city['City'],
        'Average AQI': lowest_aqi_city['Average_AQI'],
        'Data Points': lowest_aqi_city['Data_Points']
    }
])

print("Cities with the Highest and Lowest Average AQI")
display(summary_table)

## For Mumbai , Chandigarh, and Delhi + extra cities

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd


# Filter dates data as for the Chandrigarh
cities = ['Delhi', 'Bengaluru', 'Chennai',
          'Hyderabad', 'Mumbai', 'Chandigarh']

plot_data = data[data['City'].isin(cities) & data['Date'].between('2019-09-03', '2020-07-01')].dropna(subset=['Date', 'AQI']).copy()

plot_data = plot_data.sort_values('Date')

# Plot all available data points
plt.figure(figsize=(16, 8))

for city in cities:
    city_data = plot_data[plot_data['City'] == city]

    plt.plot(
        city_data['Date'],
        city_data['AQI'],
        marker='o',
        markersize=3,
        linewidth=1.5,
        label=city
    )

plt.title('AQI Trends Across Six Cities (Sep 2019 – Jul 2020)', fontsize=15)
plt.xlabel('Date', fontsize=12)
plt.ylabel('AQI', fontsize=12)
plt.legend(title='City')
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

By averaging the months data

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd


# Define cities and date range
cities = ['Delhi', 'Bengaluru', 'Chennai',
    'Hyderabad', 'Mumbai', 'Chandigarh']

start_date = pd.Timestamp('2019-09-03')
end_date = pd.Timestamp('2020-07-01')

# Filtering the given cities data
plot_data = data[data['City'].isin(cities) & data['Date'].between(start_date, end_date)].dropna(subset=['Date', 'AQI']).copy()

# Create Year-Month column and calculating monthly average AQI
plot_data['Month'] = plot_data['Date'].dt.to_period('M')

monthly_aqi = (plot_data.groupby(['City', 'Month'])['AQI'].mean().reset_index())

# Converting Month to datetime for plotting the data
monthly_aqi['Month'] = monthly_aqi['Month'].dt.to_timestamp()

# Defining the colors
colors = {
    'Delhi': '#E6194B',
    'Bengaluru': '#3CB44B',
    'Chennai': '#4363D8',
    'Hyderabad': '#F58231',
    'Mumbai': '#911EB4',
    'Chandigarh': '#469990'
}

# Plot monthly average AQI
plt.figure(figsize=(15, 7))

for city in cities:
    city_data = monthly_aqi[monthly_aqi['City'] == city].sort_values('Month')

    plt.plot(
        city_data['Month'],
        city_data['AQI'],
        marker='o',
        linewidth=2,
        markersize=5,
        label=city,
        color=colors[city]
    )

plt.title('Monthly Average AQI for Six Cities ''(Sep 2019 – Jul 2020)',fontsize=15)
plt.xlabel('Month', fontsize=12)
plt.ylabel('Monthly Average AQI', fontsize=12)

plt.xticks(
    monthly_aqi['Month'].drop_duplicates().sort_values(),
    monthly_aqi['Month'].drop_duplicates().sort_values().dt.strftime('%b %Y'),
    rotation=45
)

plt.legend(title='City')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Pollutant correlations

Pollutant values distribution

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Select pollutant features
pollutants = [
    'PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3',
    'CO', 'SO2', 'O3', 'Benzene', 'Toluene', 'Xylene'
]

# Create box plot
plt.figure(figsize=(14, 7))

sns.boxplot(data=data[pollutants])

plt.title('Box Plot of Pollutant Features', fontsize=16)
plt.xlabel('Pollutant', fontsize=12)
plt.ylabel('Concentration', fontsize=12)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pollutants = [
    'PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3',
    'CO', 'SO2', 'O3', 'Benzene', 'Toluene', 'Xylene'
]


pollutant_data = data[pollutants].copy()


correlation_matrix = pollutant_data.corr(method='pearson')

# Plotting the correlation heatmap
plt.figure(figsize=(13, 10))

sns.heatmap(
    correlation_matrix,
    annot=True,
    cmap='coolwarm',
    fmt='.2f',
    linewidths=0.5,
    vmin=-1,
    vmax=1,
    center=0,
    square=True
)

plt.title('Correlation Analysis Among Air Pollutants', fontsize=16)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## Yearly / Monthly Trends

Without averaging the month

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

cities = sorted(data['City'].unique())
years = sorted(data['Year'].unique())


year_colors = dict(
    zip(years, sns.color_palette('husl', n_colors=len(years)))
)

# Use a leap year - so that February 29th is valid
reference_year = 2020

# Create a common date using the actual month and day
data['Plot_Date'] = pd.to_datetime(
    str(reference_year) + '-' +
    data['Month'].astype(str).str.zfill(2) + '-' +
    data['Day'].astype(str).str.zfill(2)
)

for city in cities:

    # Select the city
    city_data = data[data['City'] == city].copy()

    plt.figure(figsize=(16, 8))

    for year in years:

        # Selecting the year
        year_data = city_data[city_data['Year'] == year].copy()

        if len(year_data) == 0:
            continue

        # Sort from January to December
        year_data = year_data.sort_values('Plot_Date')

        # Ploting actual daily AQI
        plt.plot(
            year_data['Plot_Date'],
            year_data['AQI'],
            label=str(year),
            color=year_colors[year],
            linewidth=1
        )

    plt.title(f'Daily AQI Across Years - {city}')
    plt.xlabel('Month')
    plt.ylabel('AQI')

    # January to December ticks for uniform date
    month_positions = pd.date_range(start=f'{reference_year}-01-01',end=f'{reference_year}-12-01',freq='MS')

    month_labels = ['January', 'February', 'March', 'April',
        'May', 'June', 'July', 'August',
        'September', 'October', 'November', 'December']

    plt.xticks(month_positions,month_labels,rotation=45)

    # Start exactly at January and end at December
    plt.xlim(pd.Timestamp(f'{reference_year}-01-01'),pd.Timestamp(f'{reference_year}-12-31'))

    plt.legend(title='Year',bbox_to_anchor=(1.02, 1),loc='upper left')

    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

With averging the month

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Create month from the original Date column
#data['Month'] = data['Date'].dt.month

# Calculating monthly AQI for each city and year
monthly_aqi = (data.groupby(['City', 'Year', 'Month'])['AQI'].mean().reset_index())

# Getting cities and years
cities = sorted(monthly_aqi['City'].unique())
years = sorted(monthly_aqi['Year'].unique())

# Creating a different color for each year
year_colors = dict(zip(years, sns.color_palette('husl', n_colors=len(years))))

# Month names
month_labels = ['January', 'February', 'March', 'April',
    'May', 'June', 'July', 'August',
    'September', 'October', 'November', 'December']

# Ploting one graph for each cities
for city in cities:

    city_data = monthly_aqi[monthly_aqi['City'] == city].copy()

    plt.figure(figsize=(14, 7))

    for year in years:

        year_data = city_data[city_data['Year'] == year].sort_values('Month')

        if len(year_data) == 0:
            continue

        plt.plot(year_data['Month'],year_data['AQI'],marker='o',linewidth=2,label=str(year),color=year_colors[year])

    plt.title(f'Monthly AQI Across Years - {city}')
    plt.xlabel('Month')
    plt.ylabel('AQI')

    plt.xticks(range(1, 13),month_labels,rotation=45)

    plt.xlim(1, 12)

    plt.legend(title='Year',bbox_to_anchor=(1.02, 1),loc='upper left')

    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
print(data.columns)

# Train - Test Spilt

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

#Pollutant features
pollutants = ['PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3','CO', 'SO2', 'O3', 'Benzene', 'Toluene', 'Xylene']


# Preparing features and target
X = data[pollutants].copy()
y = pd.to_numeric(data['AQI'], errors='coerce')


X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.20,random_state=42)


print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

print("\nInput features:")
print(X_train.columns.tolist())

# Model


## Linear Regression Model

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np


model_lr = LinearRegression()
model_lr.fit(X_train, y_train)
y_pred_lr = model_lr.predict(X_test)


print("MAE:", mean_absolute_error(y_test, y_pred_lr))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_lr)))
print("R2 Score:", r2_score(y_test, y_pred_lr))

Feature Importance

In [ ]:
import pandas as pd

importance_lr = pd.Series(
    abs(model_lr.coef_),
    index=X_train.columns
).sort_values(ascending=False)

print(importance_lr)


#To save the model and run for the future reference
'''
import joblib

# Save the trained Linear Regression model
joblib.dump(model_lr, "aqi_linear_regression.pkl")

print("Model saved successfully!")

import pandas as pd
import joblib

# Load saved model
model_lr = joblib.load("aqi_linear_regression.pkl")

# Read new data
new_data = pd.read_csv("new_aqi_data.csv")

# Select the same features used during training
X_new = new_data[X_train.columns]

# Predict AQI
new_data["Predicted_AQI"] = model_lr.predict(X_new)

# Save predictions
new_data.to_csv("aqi_predictions.csv", index=False)

print(new_data[["Predicted_AQI"]].head())
print("Predictions saved to aqi_predictions.csv")
'''

correlation filter

In [ ]:
import pandas as pd
import numpy as np

# Calculate correlation between input features
corr_matrix = X_train.corr().abs()

# Select upper triangle to avoid duplicate pairs
upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

# Set correlation threshold
threshold = 0.90

# Identify features to remove
to_drop = [
    column for column in upper.columns
    if any(upper[column] > threshold)
]

print("Highly correlated features to remove:")
print(to_drop)

# Retain selected features
X_train_filtered = X_train.drop(columns=to_drop)
X_test_filtered = X_test.drop(columns=to_drop)

print("Original number of features:", X_train.shape[1])
print("Remaining number of features:", X_train_filtered.shape[1])

## Decision Tree Model

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np


model_dt = DecisionTreeRegressor(max_depth=10, min_samples_leaf=20, min_samples_split=2)
model_dt.fit(X_train, y_train)
y_pred_dt = model_dt.predict(X_test)


print("MAE:", mean_absolute_error(y_test, y_pred_dt))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_dt)))
print("R2 Score:", r2_score(y_test, y_pred_dt))

Feature Importance

In [ ]:
importance_dt = pd.Series(
    model_dt.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

print(importance_dt)

## Random Forest Model

In [ ]:

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np


model_rf = RandomForestRegressor(n_estimators= 120, min_samples_split= 10, min_samples_leaf= 1, max_features= 0.5, max_depth= 20, bootstrap= True)
model_rf.fit(X_train, y_train)
y_pred_rf = model_rf.predict(X_test)


print("MAE:", mean_absolute_error(y_test, y_pred_rf))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_rf)))
print("R2 Score:", r2_score(y_test, y_pred_rf))


Feature importance

In [ ]:
importance = pd.Series(
    model_rf.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

print(importance)

# Next Day Predication Model

In [ ]:

import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

df = AQI_data.copy()

print("Available cities:", df["City"].dropna().unique())
city_name = input("Enter city name: ").strip()

# Find the selected city
city_df = df[df["City"].str.casefold() == city_name.casefold()].copy()

if city_df.empty:
    print("City not found. Please check the city name.")

else:
    city_df["Date"] = pd.to_datetime(
        city_df["Date"], errors="coerce"
    )
    city_df["AQI"] = pd.to_numeric(
        city_df["AQI"], errors="coerce"
    )

    city_df = city_df.dropna(subset=["Date", "AQI"])

    # Calculate daily average AQI
    city_df = (
        city_df.groupby("Date", as_index=False)["AQI"]
        .mean()
        .sort_values("Date")
        .reset_index(drop=True)
    )

    # Create lag features
    city_df["AQI_lag_1"] = city_df["AQI"].shift(1)
    city_df["AQI_lag_2"] = city_df["AQI"].shift(2)
    city_df["AQI_lag_3"] = city_df["AQI"].shift(3)
    city_df["AQI_lag_7"] = city_df["AQI"].shift(7)

    # Create rolling averages using previous observations
    city_df["AQI_rolling_mean_3"] = (
        city_df["AQI"].shift(1).rolling(window=3).mean()
    )

    city_df["AQI_rolling_mean_7"] = (
        city_df["AQI"].shift(1).rolling(window=7).mean()
    )

    features = [
        "AQI_lag_1",
        "AQI_lag_2",
        "AQI_lag_3",
        "AQI_lag_7",
        "AQI_rolling_mean_3",
        "AQI_rolling_mean_7"
    ]

    city_df["Target_AQI"] = city_df["AQI"]

    # Remove rows with missing features
    model_df = city_df.dropna(
        subset=features + ["Target_AQI"]
    ).reset_index(drop=True)

    if len(model_df) < 30:
        print("Not enough historical data for reliable evaluation.")

    else:
        X = model_df[features]
        y = model_df["Target_AQI"]
        X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.20,random_state=42)

        # Create Random Forest model
        model = RandomForestRegressor(bootstrap = True, max_depth = 15, max_features = 'log2', min_samples_leaf = 8, min_samples_split = 8, n_estimators = 100)

        # Train only on training data
        model.fit(X_train, y_train)

        # Predict AQI for the test period
        predictions = model.predict(X_test)

        # Calculate regression metrics
        mae = mean_absolute_error(y_test, predictions)
        mse = mean_squared_error(y_test, predictions)
        rmse = np.sqrt(mse)
        r2 = r2_score(y_test, predictions)

        print("\nRegression Metrics for", city_name)
        print("--------------------------------")
        print("Training samples:", len(X_train))
        print("Testing samples:", len(X_test))
        print("MAE:", round(mae, 3))
        print("MSE:", round(mse, 3))
        print("RMSE:", round(rmse, 3))
        print("R2 Score:", round(r2, 3))

        # Compare actual and predicted AQI
        results = pd.DataFrame({
            "Date": model_df.loc[X_test.index, "Date"].values,
            "Actual_AQI": y_test.values,
            "Predicted_AQI": predictions
        })

        print("\nActual vs Predicted AQI:")
        print(results.head(10))


City specific model

Regression metrics result:

1.Bengaluru

Best hyperparameters:{'n_estimators': 50, 'min_samples_split': 8, 'min_samples_leaf': 10, 'max_features': 0.7, 'max_depth': 5, 'bootstrap': True}

Best cross-validation MAE: 20.555

Test set evaluation:
MAE: 9.191
RMSE: 12.404
R2 Score: 0.6

2.Delhi

Best hyperparameters:{'n_estimators': 75, 'min_samples_split': 10, 'min_samples_leaf': 8, 'max_features': 1.0, 'max_depth': None, 'bootstrap': True}

Best cross-validation MAE: 45.572

Test set evaluation:
MAE: 32.746
RMSE: 44.657
R2 Score: 0.846

3.Hyderbad

Best hyperparameters:{'n_estimators': 150, 'min_samples_split': 15, 'min_samples_leaf': 8, 'max_features': 1.0, 'max_depth': None, 'bootstrap': True}

Best cross-validation MAE: 18.256

Test set evaluation:
MAE: 11.352
RMSE: 15.353
R2 Score: 0.814

4.Mumbai

Best hyperparameters:{'n_estimators': 125, 'min_samples_split': 10, 'min_samples_leaf': 8, 'max_features': 0.7, 'max_depth': 15, 'bootstrap': False}

Best cross-validation MAE: 20.858

Test set evaluation:
MAE: 11.803
RMSE: 18.075
R2 Score: 0.825

5.Chandigarh

Best hyperparameters:{'n_estimators': 150, 'min_samples_split': 10, 'min_samples_leaf': 2, 'max_features': 1.0, 'max_depth': 30, 'bootstrap': True}

Best cross-validation MAE: 31.064

Test set evaluation:
MAE: 12.442
RMSE: 15.69
R2 Score: 0.223

6.Chennai

Best hyperparameters:{'n_estimators': 50, 'min_samples_split': 8, 'min_samples_leaf': 10, 'max_features': 0.7, 'max_depth': 5, 'bootstrap': True}

Best cross-validation MAE: 26.975

Test set evaluation:
MAE: 18.132
RMSE: 26.04
R2 Score: 0.475

In [ ]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score



# THE DATA


df_next = data.copy()


df_next = df_next.dropna(subset=["City", "Date", "AQI"]).copy()

# Calculate daily average AQI for each city
df = (df_next.groupby(["City", "Date"], as_index=False)["AQI"].mean()
    .sort_values(["City", "Date"])
    .reset_index(drop=True)
)



#  CREATING TIME - SERIES FEATURES FOR ALL CITIES


df_next["AQI_lag_1"] = df_next.groupby("City")["AQI"].shift(1)
df_next["AQI_lag_2"] = df_next.groupby("City")["AQI"].shift(2)
df_next["AQI_lag_3"] = df_next.groupby("City")["AQI"].shift(3)
df_next["AQI_lag_7"] = df_next.groupby("City")["AQI"].shift(7)

df_next["AQI_rolling_mean_3"] = (df_next.groupby("City")["AQI"].transform(lambda s: s.shift(1).rolling(window=3).mean()))

df_next["AQI_rolling_mean_7"] = (df_next.groupby("City")["AQI"].transform(lambda s: s.shift(1).rolling(window=7).mean()))


features = [
    "AQI_lag_1",
    "AQI_lag_2",
    "AQI_lag_3",
    "AQI_lag_7",
    "AQI_rolling_mean_3",
    "AQI_rolling_mean_7"
]

#print("Features created successfully.")
#print(df[["City", "Date", "AQI"] + features].head(10))



#  DEFINING THE BEST PARAMETERS FOR EACH CITY


city_parameters = {

    "Delhi": {
        "n_estimators": 75,
        "min_samples_split": 10,
        "min_samples_leaf": 8,
        "max_features": 1.0,
        "max_depth": None,
        "bootstrap": True
    },

    "Bengaluru": {
        "n_estimators": 50,
        "min_samples_split": 8,
        "min_samples_leaf": 10,
        "max_features": 0.7,
        "max_depth": 5,
        "bootstrap": True
    },

    "Chennai": {
        "n_estimators": 50,
        "min_samples_split": 8,
        "min_samples_leaf": 10,
        "max_features": 0.7,
        "max_depth": 5,
        "bootstrap": True
    },

    "Hyderabad": {
        "n_estimators": 150,
        "min_samples_split": 15,
        "min_samples_leaf": 8,
        "max_features": 1.0,
        "max_depth": None,
        "bootstrap": True
    },

    "Mumbai": {
        "n_estimators": 125,
        "min_samples_split": 10,
        "min_samples_leaf": 8,
        "max_features": 0.7,
        "max_depth": 15,
        "bootstrap": False
    },

    "Chandigarh": {
        "n_estimators": 150,
        "min_samples_split": 10,
        "min_samples_leaf": 2,
        "max_features": 1.0,
        "max_depth": 30,
        "bootstrap": True
    }
}



#TRAINING AND STORING EACH CITY'S MODEL


trained_models = {}

for city_name, params in city_parameters.items():

    city_df = df_next[df_next["City"].str.strip().str.casefold() == city_name.casefold()].copy()

    model_df = city_df.dropna(subset=features + ["AQI"]).reset_index(drop=True)

    if len(model_df) < 20:
        print(f"Insufficient data for {city_name}.")
        continue

    X = model_df[features]
    y = model_df["AQI"]

    # Training on all available observations for this city
    model = RandomForestRegressor(
        **params,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X, y)

    # Store the model using its city name
    trained_models[city_name] = model

    #print(f"Model trained successfully for {city_name}.")



# ASK THE USER TO SELECT A CITY


print("\nAvailable cities:", ", ".join(trained_models.keys()))

city_input = input("\nEnter the city for next-day AQI prediction: ").strip()

selected_city = next(
    (
        city for city in trained_models
        if city.casefold() == city_input.casefold()
    ),
    None
)



#USING ONLY THE SELECTED CITY'S MODEL


if selected_city is None:

    print("City not found. Please enter a valid city name.")

else:

    # Get only the selected city's observations
    selected_df = df_next[df_next["City"].str.strip().str.casefold() == selected_city.casefold()].copy()

    selected_df = (selected_df.sort_values("Date").reset_index(drop=True))

    # Get the latest recorded observation
    latest_row = selected_df.iloc[-1]

    # Features were already calculated in Section 2
    latest_features = pd.DataFrame(
        [{
            feature: latest_row[feature]
            for feature in features
        }]
    )

    if latest_features.isna().any().any():

        print( f"Insufficient recent data to predict AQI for " f"{selected_city}.")

    else:

        # Retrieve ONLY the selected city's trained model
        selected_model = trained_models[selected_city]

        # Predict next-day AQI
        next_day_aqi = selected_model.predict(latest_features[features])[0]

        print("\n NEXT-DAY AQI FORECAST ")
        print("Selected city:", selected_city)
        print("Latest recorded date:", latest_row["Date"].date())
        print("Latest recorded AQI:", round(latest_row["AQI"], 2))
        print("Predicted date:",(latest_row["Date"] + pd.Timedelta(days=1)).date() )
        print("Predicted next-day AQI:", round(next_day_aqi, 2))